In [ ]:
from pathlib import Path
from pprint import pprint

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
PROJECT_ROOT = Path().resolve().parent
REPORT_PATH = PROJECT_ROOT / "report-2026-06-19-interscada-fr.joblib"

data = joblib.load(REPORT_PATH)
len(data)

655

In [ ]:
pprint(data[0])

{'cct_true': 0.8,
 'cct_weighted_global': None,
 'cct_weighted_per_location': None,
 'crit_gen_true': 'nan',
 'distance_mean': None,
 'distance_median': None,
 'distance_min': None,
 'distance_norm': None,
 'distance_spread': None,
 'has_crit_gen_prediction': False,
 'has_location_prediction': False,
 'location_neighbor_count': 0,
 'location_true': 'argoel71penl5',
 'location_weight_mass': None,
 'n_eff': None,
 'n_neighbors': 0,
 'neighborhood_compactness': None,
 'prediction_summary': None,
 'state': 'scenario4',
 'state_norm': 'scenario4'}


In [ ]:
def summary_value(summary, name: str, default=None):
    return getattr(summary, name, default) if summary is not None else default


df = pd.DataFrame(
    {
        "state": str(d["state"]),
        "cct_true": float(d["cct_true"]),
        "crit_gen_true": d["crit_gen_true"],
        "location_true": d["location_true"],
        "cct_weighted_per_location": d.get("cct_weighted_per_location"),
        "cct_weighted_global": d.get("cct_weighted_global", summary_value(d.get("prediction_summary"), "cct_weighted")),
        "has_crit_gen_prediction": d.get("has_crit_gen_prediction", d.get("prediction_summary") is not None),
        "has_location_prediction": d.get("has_location_prediction", d.get("cct_weighted_per_location") is not None),
        "location_weight_mass": d.get("location_weight_mass"),
        "location_neighbor_count": d.get("location_neighbor_count"),
        "n_neighbors": d.get("n_neighbors", summary_value(d.get("prediction_summary"), "n")),
        "n_eff": d.get("n_eff", summary_value(d.get("prediction_summary"), "n_eff")),
        "neighborhood_compactness": d.get("neighborhood_compactness"),
        "distance_min": d.get("distance_min"),
        "distance_mean": d.get("distance_mean"),
        "distance_median": d.get("distance_median"),
        "distance_spread": d.get("distance_spread"),
        "distance_norm": d.get("distance_norm"),
    }
    for d in data
)

df.head()

,state,cct_true,crit_gen_true,location_true,cct_weighted_per_location,cct_weighted_global,has_crit_gen_prediction,has_location_prediction,location_weight_mass,location_neighbor_count,n_neighbors,n_eff,neighborhood_compactness,distance_min,distance_mean,distance_median,distance_spread,distance_norm
0,scenario4,0.800000,nan,argoel71penl5,NaN,NaN,False,False,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,scenario4,0.518750,palue7paluet1,barnal71palue,NaN,NaN,False,False,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,scenario4,0.571484,palue7paluet4,barnal71penl5,NaN,NaN,False,False,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,scenario4,0.800000,palue7paluet3,barnal72palue,NaN,NaN,False,False,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,scenario4,0.612500,palue7paluet3,barnal72penl5,NaN,NaN,False,False,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
coverage = pd.Series(
    {
        "n_total": len(df),
        "n_with_crit_gen_prediction": int(df["has_crit_gen_prediction"].sum()),
        "n_with_location_prediction": int(df["has_location_prediction"].sum()),
        "crit_gen_coverage": float(df["has_crit_gen_prediction"].mean()),
        "location_coverage": float(df["has_location_prediction"].mean()),
        "n_missing_crit_gen_prediction": int((~df["has_crit_gen_prediction"]).sum()),
        "n_missing_location_prediction": int((~df["has_location_prediction"]).sum()),
    }
)

coverage

n_total                          655.000000
n_with_crit_gen_prediction        24.000000
n_with_location_prediction        24.000000
crit_gen_coverage                  0.036641
location_coverage                  0.036641
n_missing_crit_gen_prediction    631.000000
n_missing_location_prediction    631.000000
dtype: float64

In [ ]:
missing = df.loc[~df["has_location_prediction"]].copy()
missing_by_crit_gen = missing["crit_gen_true"].value_counts()
missing_by_location = missing["location_true"].value_counts().head(20)

display(missing_by_crit_gen)
display(missing_by_location)

crit_gen_true
flama7flamat1    206
palue7paluet3    153
palue7paluet4    103
palue7paluet1     52
penly7penlyt2     52
nan               40
penly7penlyt1     25
Name: count, dtype: int64

location_true
argoel71penl5    53
barnal71penl5    53
barnal72penl5    53
barnal73palue    53
flamal72menue    53
flamal73menue    53
limeul71penl5    53
barnal71palue    52
barnal72palue    52
barnal74palue    52
flamal71menue    52
flamal74menue    52
Name: count, dtype: int64

In [ ]:
def regression_metrics(frame: pd.DataFrame, pred_col: str) -> pd.Series:
    valid = frame[pred_col].notna()
    n_valid = int(valid.sum())

    if n_valid == 0:
        return pd.Series(
            {
                "n": 0,
                "coverage": 0.0,
                "mse": np.nan,
                "rmse": np.nan,
                "mae": np.nan,
                "abs_err_min": np.nan,
                "abs_err_q25": np.nan,
                "abs_err_q50": np.nan,
                "abs_err_q75": np.nan,
                "abs_err_q90": np.nan,
                "abs_err_q95": np.nan,
                "abs_err_q99": np.nan,
                "abs_err_max": np.nan,
            }
        )

    y_true = frame.loc[valid, "cct_true"].to_numpy(dtype=float)
    y_pred = frame.loc[valid, pred_col].to_numpy(dtype=float)
    err = np.abs(y_pred - y_true)

    return pd.Series(
        {
            "n": n_valid,
            "coverage": float(valid.mean()),
            "mse": mean_squared_error(y_true, y_pred),
            "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
            "mae": mean_absolute_error(y_true, y_pred),
            "abs_err_min": np.quantile(err, 0.00),
            "abs_err_q25": np.quantile(err, 0.25),
            "abs_err_q50": np.quantile(err, 0.50),
            "abs_err_q75": np.quantile(err, 0.75),
            "abs_err_q90": np.quantile(err, 0.90),
            "abs_err_q95": np.quantile(err, 0.95),
            "abs_err_q99": np.quantile(err, 0.99),
            "abs_err_max": np.quantile(err, 1.00),
        }
    )


metrics_conditional = pd.DataFrame(
    {
        "per_location": regression_metrics(df, "cct_weighted_per_location"),
        "global_fallback": regression_metrics(df, "cct_weighted_global"),
    }
)

df["cct_pred_overall"] = df["cct_weighted_per_location"].fillna(df["cct_weighted_global"])
metrics_overall = pd.DataFrame(
    {
        "location_then_global": regression_metrics(df, "cct_pred_overall"),
    }
)

metrics_conditional, metrics_overall

(             per_location  global_fallback
 n               24.000000        24.000000
 coverage         0.036641         0.036641
 mse              0.026175         0.027909
 rmse             0.161788         0.167059
 mae              0.115723         0.128052
 abs_err_min      0.011719         0.005859
 abs_err_q25      0.017579         0.020874
 abs_err_q50      0.076172         0.112061
 abs_err_q75      0.183105         0.183105
 abs_err_q90      0.304687         0.283008
 abs_err_q95      0.324609         0.297070
 abs_err_q99      0.328125         0.366504
 abs_err_max      0.328125         0.386719,
              location_then_global
 n                       24.000000
 coverage                 0.036641
 mse                      0.026175
 rmse                     0.161788
 mae                      0.115723
 abs_err_min              0.011719
 abs_err_q25              0.017579
 abs_err_q50              0.076172
 abs_err_q75              0.183105
 abs_err_q90              0.30468

In [ ]:
by_crit_gen = (
    df.groupby("crit_gen_true", observed=True)
    .apply(lambda g: regression_metrics(g, "cct_weighted_per_location"), include_groups=False)
    .sort_values("mae")
    .astype({"n": int})
)

by_crit_gen.to_csv("./interscada_fr_report.csv")

by_crit_gen

,n,coverage,mse,rmse,mae,abs_err_min,abs_err_q25,abs_err_q50,abs_err_q75,abs_err_q90,abs_err_q95,abs_err_q99,abs_err_max
crit_gen_true,,,,,,,,,,,,,
palue7paluet1,2,0.037037,0.000137,0.011719,0.011719,0.011719,0.011719,0.011719,0.011719,0.011719,0.011719,0.011719,0.011719
flama7flamat1,8,0.037383,0.000369,0.019212,0.019043,0.017579,0.017579,0.017579,0.019043,0.023437,0.023437,0.023437,0.023437
palue7paluet3,6,0.037736,0.036587,0.191276,0.150390,0.017578,0.045410,0.128906,0.260742,0.304687,0.304687,0.304687,0.304687
penly7penlyt2,2,0.037037,0.028873,0.169922,0.169922,0.169922,0.169922,0.169922,0.169922,0.169922,0.169922,0.169922,0.169922
penly7penlyt1,2,0.074074,0.049576,0.222656,0.222656,0.222656,0.222656,0.222656,0.222656,0.222656,0.222656,0.222656,0.222656
palue7paluet4,4,0.037383,0.062141,0.249282,0.228516,0.128906,0.128906,0.228516,0.328125,0.328125,0.328125,0.328125,0.328125
nan,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
missing_diagnostics = {
    "missing_by_crit_gen": df.loc[~df["has_crit_gen_prediction"], "crit_gen_true"].value_counts().head(20),
    "missing_by_location": df.loc[~df["has_location_prediction"], "location_true"].value_counts().head(20),
}

missing_diagnostics

{'missing_by_crit_gen': crit_gen_true
 flama7flamat1    206
 palue7paluet3    153
 palue7paluet4    103
 palue7paluet1     52
 penly7penlyt2     52
 nan               40
 penly7penlyt1     25
 Name: count, dtype: int64,
 'missing_by_location': location_true
 argoel71penl5    53
 barnal71penl5    53
 barnal72penl5    53
 barnal73palue    53
 flamal72menue    53
 flamal73menue    53
 limeul71penl5    53
 barnal71palue    52
 barnal72palue    52
 barnal74palue    52
 flamal71menue    52
 flamal74menue    52
 Name: count, dtype: int64}

In [ ]:
from scipy.stats import spearmanr


def analysis():
    _df = pd.DataFrame(data)
    _df = _df.drop(columns=["prediction_summary"])
    _df = _df.dropna(subset=["cct_weighted_per_location"])
    _df["err"] = (_df["cct_true"] - _df["cct_weighted_per_location"]).abs()

    rho, p_value = spearmanr(_df.err, _df.n_eff)
    print("n_eff", f"{rho=} {p_value=}")

    rho, p_value = spearmanr(_df.err, _df.neighborhood_compactness)
    print("neighborhood_compactness", f"{rho=} {p_value=}")

    rho, p_value = spearmanr(_df.err, _df.n_neighbors)
    print("n_neighbors", f"{rho=} {p_value=}")

    rho, p_value = spearmanr(_df.err, _df.location_weight_mass)
    print("location_weight_mass", f"{rho=} {p_value=}")

    dist_cols = [c for c in _df.columns if c.startswith("distance")]
    for col in dist_cols:
        rho, p_value = spearmanr(_df.err, _df[col])
        print(col, f"{rho=} {p_value=}")


analysis()

n_eff rho=np.float64(-0.33921510645761843) p_value=np.float64(0.1048907248462228)
neighborhood_compactness rho=nan p_value=nan
n_neighbors rho=np.float64(-0.34135228962833236) p_value=np.float64(0.10258077938033293)
location_weight_mass rho=np.float64(0.33921510645761843) p_value=np.float64(0.1048907248462228)
distance_min rho=np.float64(0.0) p_value=np.float64(1.0)
distance_mean rho=np.float64(0.009590156923103561) p_value=np.float64(0.9645259709795422)
distance_median rho=np.float64(0.0) p_value=np.float64(1.0)
distance_spread rho=nan p_value=nan
distance_norm rho=nan p_value=nan


/tmp/ipykernel_1498443/3860746159.py:24: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, p_value = spearmanr(_df.err, _df[col])


In [ ]:
_df = pd.DataFrame(data)
_df = _df.drop(columns=["prediction_summary"])
_df = _df.dropna(subset=["cct_weighted_per_location"])
_df["err"] = (_df["cct_true"] - _df["cct_weighted_per_location"]).abs()

In [ ]:
def risk_coverage(df, metric, higher_is_better=True, coverages=(1.0, 0.95, 0.9, 0.8, 0.7, 0.5)):
    x = df.dropna(subset=[metric, "err"]).copy()
    x = x.sort_values(metric, ascending=not higher_is_better)

    rows = []
    n = len(x)
    for cov in coverages:
        k = int(np.ceil(cov * n))
        kept = x.iloc[:k]
        rows.append(
            {
                "metric": metric,
                "coverage": cov,
                "n": k,
                "mae": kept["err"].mean(),
                "rmse": np.sqrt((kept["err"] ** 2).mean()),
                "q90": kept["err"].quantile(0.90),
                "q95": kept["err"].quantile(0.95),
            }
        )
    return pd.DataFrame(rows)


metrics = {
    "location_weight_mass": True,
    "n_eff": True,
    "n_neighbors": True,
    "neighborhood_compactness": True,
    "distance_min": False,
    "distance_mean": False,
    "distance_median": False,
    "distance_spread": False,
    "distance_norm": False,
}

out = []
for metric, higher_is_better in metrics.items():
    out.append(risk_coverage(_df, metric, higher_is_better))

rc = pd.concat(out, ignore_index=True)
rc.to_csv("./risk_coverage_interscada_fr.csv")
rc

,metric,coverage,n,mae,rmse,q90,q95
0,location_weight_mass,1.00,24,0.115723,0.161788,0.304687,0.324609
1,location_weight_mass,0.95,23,0.119990,0.165227,0.304687,0.325781
2,location_weight_mass,0.90,22,0.124645,0.168899,0.304687,0.326953
3,location_weight_mass,0.80,20,0.135059,0.177022,0.307031,0.328125
4,location_weight_mass,0.70,17,0.155790,0.191865,0.314062,0.328125
5,location_weight_mass,0.50,12,0.170410,0.203896,0.325781,0.328125
6,n_eff,1.00,24,0.115723,0.161788,0.304687,0.324609
7,n_eff,0.95,23,0.113366,0.161425,0.304687,0.325781
8,n_eff,0.90,22,0.110796,0.161029,0.304687,0.326953
9,n_eff,0.80,20,0.110156,0.161362,0.307031,0.328125
